# OTTER methodology

OTTER fits a semirelaxed fused Gromov–Wasserstein coupling. The objective contains a cross-species cost and a relational connectivity cost:

$$
\\pi^* = \\arg\\min_{\\pi \\ge 0,\\; \\pi\\mathbf{1}=p}
(1-\\alpha)\\langle M,\\pi\\rangle +
\\alpha \\sum_{i,j,k,l}(C_m[i,k]-C_h[j,l])^2\\pi[i,j]\\pi[k,l].
$$

There is no entropy term in this objective. The solver uses epsilon as the weight of a KL proximal penalty between successive iterates. The released coupling is the 25th iterate at epsilon = 0.05.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from otter.data import load_pi, pi_provenance
from otter.repro import CANONICAL, anchor_warped_xyz, fit_coupling, load_inputs

mouse, human, costs, regional_entries = load_inputs(ROOT)
print(f'mouse parcels: {len(mouse.var):,}')
print(f'human parcels: {len(human.var):,}')
print(f'regional entries: {len(regional_entries)}')
print(CANONICAL)

## Canonical costs

The relational cost mixes functional and structural connectivity at 0.7:0.3. The cross-species cost contains an anchor-warped spatial distance, 21 Garin homology classes and 26 curated regional entries. Gene expression is not used to fit the canonical coupling.

In [ ]:
spatial_cost = anchor_warped_xyz(mouse, human)
print(spatial_cost.shape, float(spatial_cost.min()), float(spatial_cost.max()))

## Hyperparameter evaluation

A 25-cell grid over epsilon and spatial weight was evaluated with one five-fold split of the 19 scorable Beauchamp correspondences. The modal fold-wise choice was spatial weight 0.25 and epsilon 0.2. The released coupling uses spatial weight 0.25 and epsilon 0.05: its benchmark accuracy was nearly identical, while its rows were more concentrated at the fixed 25-iteration stopping point. The release should therefore be described as informed by the grid and a concentration trade-off, not as the unique cross-validation optimum.

In [ ]:
sweep = json.loads((ROOT / 'outputs/logs/section5_canonical_sweep.json').read_text())
modal = sweep['nested_cv']['modal_selected_cell']
deployed = sweep['deploy']['cell']
print('fold-wise selections:', sweep['nested_cv']['per_fold_selected'])
print('modal cell:', modal)
print('released cell:', deployed)
print('all-pair top-1, modal:', sweep['cells'][modal]['beauchamp_top1'])
print('all-pair top-1, released:', sweep['cells'][deployed]['beauchamp_top1'])

## Optional refit

The released coupling should normally be loaded rather than recomputed. Set RUN_REFIT only when checking the fitting implementation; save the result under a new filename and compare its provenance before using existing result logs.

In [ ]:
RUN_REFIT = False

if RUN_REFIT:
    pi_refit = fit_coupling(
        mouse, human, costs, regional_entries, spatial_cost, **CANONICAL
    )
    np.save(ROOT / 'outputs/coupling/pi_canonical_refit.npy', pi_refit)
else:
    pi_release = load_pi()
    print(pi_release.shape)
    print(pi_provenance())